# 06 — Molecular Docking of Predicted EGFR Inhibitors

## Objective

This notebook evaluates the binding potential of 10 candidate EGFR inhibitors identified through machine-learning-guided virtual screening of the ZINC22 compound library.

The candidates were previously prioritized based on predicted pIC50, Lipinski drug-likeness criteria, structural similarity to the EGFR training dataset, and molecular diversity.

Molecular docking will be performed using AutoDock Vina to investigate predicted binding orientations, binding affinities, and protein–ligand interactions within the EGFR kinase binding pocket.

## Workflow

1. **Protein Structure Retrieval:** Obtain and inspect the EGFR crystal structure (PDB: 6LUD).
2. **Receptor Preparation:** Prepare the protein structure for molecular docking.
3. **Ligand Preparation:** Generate and optimize 3D structures of the 10 selected ZINC compounds.
4. **Binding Site Identification:** Define the docking region using the experimentally observed inhibitor-binding pocket.
5. **Docking Validation:** Evaluate the docking protocol using an appropriate reference ligand.
6. **Molecular Docking:** Dock the selected compounds using AutoDock Vina.
7. **Results Analysis:** Compare predicted binding affinities, binding poses, and key protein–ligand interactions.

## Expected Outputs

- Prepared EGFR receptor and ligand structures
- Docking scores for all 10 candidate compounds
- Predicted protein–ligand binding poses
- Visualizations of selected protein–ligand interactions
- Comparison of molecular docking results with machine-learning predictions

**Note:** PDB 6LUD represents a mutant EGFR structure containing L858R/T790M/C797S substitutions and the covalent inhibitor osimertinib. Conventional AutoDock Vina docking evaluates noncovalent interactions; therefore, docking scores will be interpreted as computational estimates rather than experimentally validated binding affinities.

In [1]:
from pathlib import Path
from collections import Counter
import urllib.request

PROJECT_ROOT = Path("..").resolve()

RECEPTOR_DIR = PROJECT_ROOT / "docking" / "input"
RECEPTOR_DIR.mkdir(parents=True, exist_ok=True)

pdb_path = RECEPTOR_DIR / "6LUD.pdb"

if not pdb_path.exists():
    urllib.request.urlretrieve(
        "https://files.rcsb.org/download/6LUD.pdb",
        pdb_path
    )

print("Structure:", pdb_path)
print("File size:", pdb_path.stat().st_size, "bytes")

with open(pdb_path) as f:
    lines = f.readlines()

chains = sorted({
    line[21].strip()
    for line in lines
    if line.startswith("ATOM  ")
})

ligands = Counter(
    (line[17:20].strip(), line[21].strip())
    for line in lines
    if line.startswith("HETATM")
    and line[17:20].strip() not in {"HOH"}
)

print("Protein chains:", chains)
print("Non-water hetero residues:", ligands)

Structure: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/6LUD.pdb
File size: 242109 bytes
Protein chains: ['A']
Non-water hetero residues: Counter({('YY3', 'A'): 37})


## 2. Receptor and Reference Ligand Extraction

The EGFR crystal structure (PDB: 6LUD) was inspected to identify its protein chain and co-crystallized ligand. Protein atoms from chain A and the reference ligand (YY3) were extracted into separate PDB files for subsequent receptor preparation and binding-site analysis.

In [2]:
# Define output paths
protein_path = RECEPTOR_DIR / "egfr_protein.pdb"
ligand_path = RECEPTOR_DIR / "egfr_reference_YY3.pdb"

# Extract protein atoms from chain A
protein_lines = [
    line for line in lines
    if line.startswith("ATOM  ") and line[21] == "A"
]

# Extract co-crystallized ligand YY3
ligand_lines = [
    line for line in lines
    if line.startswith("HETATM")
    and line[17:20].strip() == "YY3"
    and line[21] == "A"
]

# Save extracted structures
with open(protein_path, "w") as f:
    f.writelines(protein_lines)
    f.write("END\n")

with open(ligand_path, "w") as f:
    f.writelines(ligand_lines)
    f.write("END\n")

print("Protein atoms:", len(protein_lines))
print("Ligand atoms:", len(ligand_lines))

print("\nSaved protein:", protein_path)
print("Saved reference ligand:", ligand_path)

Protein atoms: 2388
Ligand atoms: 37

Saved protein: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_protein.pdb
Saved reference ligand: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_reference_YY3.pdb


## 3. Receptor Structure Inspection

The extracted EGFR protein structure was examined to identify its residue composition, alternate atomic conformations, and potential structural issues before receptor preparation. This inspection was performed to assess the suitability of the crystallographic structure for molecular docking.

In [ ]:
# %pip install biopython

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 20.8 MB/s  0:00:01 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for biopython: filename=biopython-1.88-cp312-cp312-macosx_11_0_x86_64.whl size=2680697 sha256=22edecb9e6f018c8a2c9159da2e03f29ba9c68cbdad47741912291eb270dbb78
  Stored in directory: /Users/gracescanlan/Library/Caches/pip/wheels/6d/bf/1b/7e7d437c3c04950fc117df31f242ad764ec5d197969f15f6ed
Successfully built biopython
Note: you may need to restart the kernel to use updated packages.


In [5]:
from Bio.PDB import PDBParser
from collections import Counter

parser = PDBParser(QUIET=True)

structure = parser.get_structure(
    "EGFR",
    str(protein_path)
)

model = structure[0]
chain = model["A"]

residues = list(chain.get_residues())
atoms = list(chain.get_atoms())

print("Protein chain:", chain.id)
print("Number of residues:", len(residues))
print("Number of atoms:", len(atoms))

# Residue numbering
residue_numbers = [
    residue.id[1]
    for residue in residues
]

print(
    "Residue range:",
    min(residue_numbers),
    "-",
    max(residue_numbers)
)

# Identify gaps in residue numbering
missing_positions = sorted(
    set(range(min(residue_numbers), max(residue_numbers) + 1))
    - set(residue_numbers)
)

print("Missing residue positions:", missing_positions)

# Identify alternate conformations
alternate_atoms = [
    atom
    for atom in atoms
    if atom.is_disordered()
]

print("Atoms with alternate conformations:", len(alternate_atoms))

# Amino acid composition
residue_counts = Counter(
    residue.resname
    for residue in residues
)

print("\nResidue composition:")
print(residue_counts)

Protein chain: A
Number of residues: 310
Number of atoms: 2388
Residue range: 696 - 1018
Missing residue positions: [748, 749, 750, 751, 991, 992, 993, 994, 995, 996, 997, 998, 999]
Atoms with alternate conformations: 0

Residue composition:
Counter({'LEU': 32, 'ILE': 25, 'GLU': 23, 'VAL': 23, 'ALA': 22, 'LYS': 22, 'ASP': 20, 'GLY': 19, 'PRO': 16, 'ARG': 15, 'SER': 15, 'MET': 13, 'TYR': 12, 'GLN': 10, 'THR': 10, 'HIS': 8, 'ASN': 7, 'PHE': 7, 'TRP': 6, 'CYS': 5})


## 4. Binding-Site Inspection

The EGFR ligand-binding pocket was characterized using the coordinates of the co-crystallized reference ligand (YY3). Protein residues within 5 Å of the ligand were identified to determine the local binding-site environment and assess whether unresolved regions could affect subsequent molecular docking.

In [8]:
import numpy as np
from Bio.PDB import PDBParser
from scipy.spatial.distance import cdist
import pandas as pd

parser = PDBParser(QUIET=True)

protein = parser.get_structure("EGFR", str(protein_path))
ligand = parser.get_structure("YY3", str(ligand_path))

# Extract ligand heavy-atom coordinates
ligand_coords = np.array([
    atom.coord
    for atom in ligand.get_atoms()
    if atom.element != "H"
])

# Calculate geometric center of the ligand
ligand_center = ligand_coords.mean(axis=0)

print("Ligand center (Å):", np.round(ligand_center, 3))

# Identify protein residues within 5 Å
binding_residues = []

for residue in protein.get_residues():

    residue_coords = np.array([
        atom.coord
        for atom in residue.get_atoms()
        if atom.element != "H"
    ])

    if len(residue_coords) == 0:
        continue

    distances = cdist(residue_coords, ligand_coords)

    min_distance = distances.min()

    if min_distance <= 5.0:
        binding_residues.append({
            "Residue": residue.resname,
            "Position": residue.id[1],
            "Min Distance (Å)": round(float(min_distance), 2)
        })

binding_df = pd.DataFrame(binding_residues)

binding_df = binding_df.sort_values("Position")

display(binding_df)

print("\nBinding-site residues:", len(binding_df))

Ligand center (Å): [-4.9731e+01 -3.9000e-02 -1.8220e+01]


,Residue,Position,Min Distance (Å)
0,LEU,718,3.60
1,GLY,719,4.08
2,PHE,723,3.58
3,VAL,726,3.41
4,LYS,728,4.68
5,ALA,743,3.55
6,LYS,745,4.11
7,MET,790,3.98
8,GLN,791,3.43
9,LEU,792,3.56



Binding-site residues: 19


## 5. Docking Grid Definition

The molecular docking search space was defined using the coordinates of the co-crystallized ligand YY3. The geometric center of the ligand was used as the docking-box center, and the box dimensions were calculated from the ligand's spatial extent with additional padding to accommodate ligand flexibility and structural variation among screened compounds.

In [9]:
import numpy as np
import pandas as pd

# Ligand coordinates calculated in Step 4
ligand_min = ligand_coords.min(axis=0)
ligand_max = ligand_coords.max(axis=0)

# Define docking box
BOX_PADDING = 8.0  # Å, added to each side

box_center = (ligand_min + ligand_max) / 2
box_size = (ligand_max - ligand_min) + 2 * BOX_PADDING

docking_box = pd.DataFrame({
    "Axis": ["X", "Y", "Z"],
    "Center (Å)": np.round(box_center, 3),
    "Size (Å)": np.round(box_size, 3)
})

display(docking_box)

print("\nDocking box dimensions calculated.")

,Axis,Center (Å),Size (Å)
0,X,-49.874001,22.877001
1,Y,0.026000,24.719000
2,Z,-18.135000,29.917999



Docking box dimensions calculated.


## 6. Receptor Preparation

The extracted EGFR receptor will be prepared for molecular docking by assigning appropriate hydrogen atoms, atomic charges, and AutoDock-compatible atom types. The prepared receptor will be exported in PDBQT format for use with AutoDock Vina.

In [10]:
import importlib.util
import shutil
import sys

print("Python executable:", sys.executable)

packages = {
    "RDKit": "rdkit",
    "Meeko": "meeko",
    "AutoDock Vina": "vina",
    "PDBFixer": "pdbfixer",
    "OpenMM": "openmm",
    "Biopython": "Bio"
}

print("\nPython packages:")
for name, module in packages.items():
    available = importlib.util.find_spec(module) is not None
    print(f"{name}: {'Installed' if available else 'Not installed'}")

print("\nCommand-line tools:")
for command in ["mk_prepare_receptor.py", "mk_prepare_ligand.py", "vina", "obabel"]:
    print(f"{command}: {shutil.which(command) or 'Not found'}")

Python executable: /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/python

Python packages:
RDKit: Installed
Meeko: Not installed
AutoDock Vina: Not installed
PDBFixer: Not installed
OpenMM: Not installed
Biopython: Installed

Command-line tools:
mk_prepare_receptor.py: Not found
mk_prepare_ligand.py: Not found
vina: Not found
obabel: Not found


In [11]:
import importlib.util
import shutil

for name in ["vina", "meeko", "pdbfixer", "openmm"]:
    print(
        name,
        "Installed" if importlib.util.find_spec(name) else "Missing"
    )

for command in ["vina", "mk_prepare_receptor.py", "mk_prepare_ligand.py", "obabel"]:
    print(command, shutil.which(command))

vina Installed
meeko Installed
pdbfixer Installed
openmm Installed
vina /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/vina
mk_prepare_receptor.py /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/mk_prepare_receptor.py
mk_prepare_ligand.py /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/mk_prepare_ligand.py
obabel /usr/local/Caskroom/miniconda/base/envs/egfr-drug-discovery/bin/obabel


## 7. Protein Structure Preparation

The extracted EGFR receptor was examined using PDBFixer to identify missing residues, missing heavy atoms, and nonstandard amino acids. Structural corrections were evaluated before hydrogen addition and conversion to AutoDock-compatible PDBQT format. The original crystallographic structure was retained for reference.

In [15]:
from pdbfixer import PDBFixer
from pathlib import Path

# Load extracted EGFR receptor
fixer = PDBFixer(filename=str(protein_path))

# Identify structural issues
fixer.findMissingResidues()
fixer.findNonstandardResidues()
fixer.findMissingAtoms()

print("Missing residues:")
print(fixer.missingResidues)

print("\nNonstandard residues:")
print(fixer.nonstandardResidues)

print("\nResidues with missing heavy atoms:")
for residue, atoms in fixer.missingAtoms.items():
    print(
        residue.name,
        residue.id,
        [atom.name for atom in atoms]
    )

Missing residues:
{}

Nonstandard residues:
[]

Residues with missing heavy atoms:
GLU 697 ['CG', 'CD', 'OE1', 'OE2']
LYS 708 ['CE', 'NZ']
LYS 713 ['CE', 'NZ']
LYS 716 ['CD', 'CE', 'NZ']
GLU 734 ['CG', 'CD', 'OE1', 'OE2']
LYS 737 ['CG', 'CD', 'CE', 'NZ']
LYS 754 ['CG', 'CD', 'CE', 'NZ']
LYS 757 ['CG', 'CD', 'CE', 'NZ']
GLU 758 ['CG', 'CD', 'OE1', 'OE2']
LYS 806 ['CG', 'CD', 'CE', 'NZ']
GLU 872 ['CG', 'CD', 'OE1', 'OE2']
LYS 875 ['CG', 'CD', 'CE', 'NZ']
GLU 922 ['CG', 'CD', 'OE1', 'OE2']
LYS 960 ['CE', 'NZ']
ARG 973 ['CZ', 'NH1', 'NH2']
GLN 982 ['CG', 'CD', 'OE1', 'NE2']
GLU 985 ['CG', 'CD', 'OE1', 'OE2']
ARG 986 ['CG', 'CD', 'NE', 'CZ', 'NH1', 'NH2']
HIS 988 ['CG', 'ND1', 'CE1', 'NE2', 'CD2']
ASP 1003 ['CG', 'OD1', 'OD2']
GLU 1004 ['CG', 'CD', 'OE1', 'OE2']
GLU 1005 ['CG', 'CD', 'OE1', 'OE2']


In [16]:
# Summarize missing structural components

print("Missing residue regions:")
for key, residue_names in fixer.missingResidues.items():
    print(key, residue_names)

print("\nNonstandard residues:")
print(fixer.nonstandardResidues)

print("\nResidues with missing heavy atoms:")
print(len(fixer.missingAtoms))

print("\nMissing heavy atoms near binding pocket:")

pocket_positions = set(binding_df["Position"])

for residue, atoms in fixer.missingAtoms.items():
    if int(residue.id) in pocket_positions:
        print(
            residue.name,
            residue.id,
            [atom.name for atom in atoms]
        )

print("\nMissing terminal atoms:")
for residue, atom_names in fixer.missingTerminals.items():
    print(residue.name, residue.id, atom_names)

Missing residue regions:

Nonstandard residues:
[]

Residues with missing heavy atoms:
22

Missing heavy atoms near binding pocket:

Missing terminal atoms:
ILE 1018 ['OXT']


## 8. Receptor Repair and Hydrogen Addition

Missing heavy atoms were reconstructed using PDBFixer while retaining the experimentally resolved protein structure. Hydrogen atoms were subsequently added at physiological pH (7.4). The prepared receptor was saved as a PDB file for subsequent conversion to AutoDock-compatible PDBQT format.

In [17]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile
from pathlib import Path

# Start from the original extracted protein
fixer = PDBFixer(filename=str(protein_path))

fixer.findMissingResidues()

# Do not automatically reconstruct missing residue segments
fixer.missingResidues = {}

fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()

fixer.findMissingAtoms()

# Reconstruct missing heavy atoms
fixer.addMissingAtoms()

# Add hydrogens at physiological pH
fixer.addMissingHydrogens(7.4)

# Save prepared receptor
prepared_path = (
    PROJECT_ROOT
    / "docking"
    / "input"
    / "egfr_prepared.pdb"
)

prepared_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(prepared_path, "w") as f:
    PDBFile.writeFile(
        fixer.topology,
        fixer.positions,
        f,
        keepIds=True
    )

print("Prepared receptor saved:", prepared_path)

print(
    "Total atoms:",
    fixer.topology.getNumAtoms()
)

print(
    "Total residues:",
    fixer.topology.getNumResidues()
)

Prepared receptor saved: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/docking/input/egfr_prepared.pdb
Total atoms: 4981
Total residues: 310
